# Data Cleaning

## Setup

In [10]:
!python --version

Python 3.12.4


In [12]:
!python -m pip install -q pandas numpy matplotlib seaborn wordcloud nltk spacy textblob scikit-learn bertopic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
label-studio 1.16.0 requires pandas>=2.2.3, but you have pandas 2.2.2 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.4.25 which is incompatible.
langchain-aws 0.2.10 requires boto3>=1.35.74, but you have boto3 1.35.29 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
langchain 0.3.14 requires langsmith<0.3,>=0.1.17, but you have langsmith 0.4.25 which is incompatible.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.0 which is incompatible.
streamlit 1.38.0 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.


### Imports

In [13]:
import pandas as pd

### Configurations

In [5]:
pd.set_option('display.max_columns', None)

In [2]:
data_path = '../data/NTSB_database.csv'

## Load the Data

In [3]:
df = pd.read_csv(data_path, low_memory=False)
df.shape

(87951, 45)

In [6]:
df.head()

,Event Id,Investigation Type,Country,Aircraft Damage,Aircraft Category,Make,Model,Amateur Built,Number Of Engines,Engine Type,Far Description,Schedule,Purpose Of Flight,Total Fatal Injuries,Total Serious Injuries,Total Minor Injuries,Total Uninjured,Weather Condition,Broad Phase Of Flight,Analysis,City,Longitude,Latitude,Address,geometry,Place,Number Of Seats,Type Aircraft,Type Engine,Total Person,Far Description Factorized,Schedule Factorized,Purpose Of Flight Factorized,Make Factorized,Model Factorized,Event Year,Publication Year,Event Month,Publication Month,Event Day,Publication Day,Date Difference,Publication Month Name,Event Month Name,Season
0,20001218X45444,Accident,United States,Destroyed,fixed wing single engine,stinson,108-3,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,UNK,Cruise,"\n \n ON OCTOBER 24, 1948, T...",moose creek,-147.160665,64.713512,"Moose Creek, Fairbanks North Star, Alaska, Uni...",POINT (-147.1606646266588 64.7135123),mountain,4,4,1,2,1,2,0,0,0,1948,2001,10,8,24,24.0,26,August,October,Fall
1,20001218X45447,Accident,United States,Destroyed,weight-shift-control,piper,pa24-180,No,1,reciprocating,part 91: general aviation,UNK,Personal,4,0,0,0,UNK,Unknown,"\n \n ON JULY 19, 1962, A CO...",bridgeport,-73.188786,41.179269,"Bridgeport, Greater Bridgeport Planning Region...",POINT (-73.1887863 41.1792695),sea,4,7,1,4,1,2,0,1,1,1962,1996,7,9,19,19.0,34,September,July,Summer
2,20061025X01555,Accident,United States,Destroyed,fixed wing single engine,cessna,172m,No,1,reciprocating,part 91: general aviation,UNK,Personal,3,0,0,1,IMC,Cruise,\n \n The private pilot was ...,saltville,-81.762063,36.881503,"Saltville, Smyth County, Virginia, United States",POINT (-81.7620635 36.8815031),sea,4,4,1,3,1,2,0,2,2,1974,2007,8,2,30,30.0,33,February,August,Summer
3,20001218X45448,Accident,United States,Destroyed,weight-shift-control,rockwell,112,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,IMC,Cruise,\n \n The aircraft wreckage ...,eureka,-124.167375,40.790687,"Eureka, Humboldt County, California, United St...",POINT (-124.1673746 40.7906871),airport,4,7,1,2,1,2,0,3,3,1977,2000,6,12,19,19.0,23,December,June,Summer
4,20041105X01764,Accident,United States,Destroyed,fixed wing multi engine,cessna,501,No,2,turbo fan,part 91: general aviation,UNK,Personal,1,2,0,0,VMC,Approach,\n \n The Safety Board's ful...,canton,-95.864051,32.555664,"Canton, Van Zandt County, Texas, 75103, United...",POINT (-95.8640507 32.555664),airport,8,5,5,3,1,2,0,2,4,1979,1980,8,4,2,2.0,1,April,August,Summer


### Normalize Column Names

In [22]:
df = df.rename({col: col.strip().lower().replace(' ', '_') for col in df.columns}, axis=1)
df.columns

Index(['event_id', 'investigation_type', 'country', 'aircraft_damage',
       'aircraft_category', 'make', 'model', 'amateur_built',
       'number_of_engines', 'engine_type', 'far_description', 'schedule',
       'purpose_of_flight', 'total_fatal_injuries', 'total_serious_injuries',
       'total_minor_injuries', 'total_uninjured', 'weather_condition',
       'broad_phase_of_flight', 'analysis', 'city', 'longitude', 'latitude',
       'address', 'geometry', 'place', 'number_of_seats', 'type_aircraft',
       'type_engine', 'total_person', 'far_description_factorized',
       'schedule_factorized', 'purpose_of_flight_factorized',
       'make_factorized', 'model_factorized', 'event_year', 'publication_year',
       'event_month', 'publication_month', 'event_day', 'publication_day',
       'date_difference', 'publication_month_name', 'event_month_name',
       'season'],
      dtype='object')

## Analyze the Data

### Missing Values

In [23]:
missing_values = df.isna().sum()
missing_ratio = missing_values / len(df)
missing_percent = missing_ratio.map(lambda x: f'{x*100:.2f}%')
missing_values_report = pd.DataFrame({
    'Missing Values': missing_values,
    'Total Values': len(df),
    'Missing Ratio': missing_ratio,
    'Missing Percent': missing_percent,
})
missing_values_report

,Missing Values,Total Values,Missing Ratio,Missing Percent
event_id,0,87951,0.000000,0.00%
investigation_type,0,87951,0.000000,0.00%
country,0,87951,0.000000,0.00%
aircraft_damage,0,87951,0.000000,0.00%
aircraft_category,0,87951,0.000000,0.00%
make,0,87951,0.000000,0.00%
model,0,87951,0.000000,0.00%
amateur_built,0,87951,0.000000,0.00%
number_of_engines,0,87951,0.000000,0.00%
engine_type,0,87951,0.000000,0.00%


**Observation:** Dataset has very few missing values and there really is no need to fill in any of the missing values.

### Uniqueness

In [24]:
unique_values = df.nunique()
unique_ratio = unique_values / len(df)
unique_percent = unique_ratio.map(lambda x: f'{x*100:.2f}%')
unique_report = pd.DataFrame({
    'Unique Values': unique_values,
    'Total Values': len(df),
    'Unique Ratio': unique_ratio,
    'Unique Percent': unique_percent,
})
unique_report

,Unique Values,Total Values,Unique Ratio,Unique Percent
event_id,87951,87951,1.000000,100.00%
investigation_type,2,87951,0.000023,0.00%
country,219,87951,0.002490,0.25%
aircraft_damage,14,87951,0.000159,0.02%
aircraft_category,21,87951,0.000239,0.02%
make,7552,87951,0.085866,8.59%
model,11563,87951,0.131471,13.15%
amateur_built,2,87951,0.000023,0.00%
number_of_engines,7,87951,0.000080,0.01%
engine_type,13,87951,0.000148,0.01%


**Observation:** One thing to note is that there are some features that have a low percentage of unique values, indicating a lot of duplicate values. This is permissable for some features but for `analysis`, which represents the report narratives, it's not good to have duplicates for NLP analysis. These duplicates will need to be filtered out.

In [25]:
# Inspect first narrative for context and baseline
print(df['analysis'].iloc[0])


          
          ON OCTOBER 24, 1948, THE PILOT DEPARTED A PRIVATE AIRSTRIP AT THE MOOSE CREEK RANCH AT 1645. TWO OTHER PILOTS THAT DEPARTED AT THE SAME TIME REPORTED ENCOUNTERING HEAVY SNOW SQUALLS ALONG THE SELWAY RIVER AS THEY HEADED WESTBOUND. THE WRECKAGE WAS FOUND IN APRIL, 1987, 19 MILES WEST OF MOOSE CREEK, IN THE SELWAY RIVER DRAINAGE, IN MOUNTAINOUS TERRAIN. THE WRECKAGE EXHIBITED CHARACTERISTICS CONSISTENT WITH IMPACT WITH TREES AND TERRAIN IN CRUISE FLIGHT.
          
          


### Clean the Data

In [26]:
# Cleanse report narratives for data curation
# 1) Remove all newline characters, carriage return characters, and tab characters in reports
# 2) Remove all records where both the primary and secondary report narratives are null
df_clean = df.replace(r"\n|\r|\t", " ", regex=True).copy(deep=True)
df_clean = df_clean.dropna(subset=['analysis'], how="all")

In [29]:
# Remove unnecessary whitespace from narrative
df_clean['analysis'] = df_clean['analysis'].apply(lambda x: x.strip())

In [31]:
# Inspect first narrative to see results of cleaning
print(df_clean['analysis'].iloc[0])

ON OCTOBER 24, 1948, THE PILOT DEPARTED A PRIVATE AIRSTRIP AT THE MOOSE CREEK RANCH AT 1645. TWO OTHER PILOTS THAT DEPARTED AT THE SAME TIME REPORTED ENCOUNTERING HEAVY SNOW SQUALLS ALONG THE SELWAY RIVER AS THEY HEADED WESTBOUND. THE WRECKAGE WAS FOUND IN APRIL, 1987, 19 MILES WEST OF MOOSE CREEK, IN THE SELWAY RIVER DRAINAGE, IN MOUNTAINOUS TERRAIN. THE WRECKAGE EXHIBITED CHARACTERISTICS CONSISTENT WITH IMPACT WITH TREES AND TERRAIN IN CRUISE FLIGHT.
